# MentalBERT fine-tuning (dev set only)
Run on **Colab with a GPU** (Runtime > Change runtime type > T4 GPU).
Upload `train_clean.csv` and `dev_clean.csv` from the cleaning notebook first.
The test set is not touched here.

**One-time setup:** MentalBERT is gated on Hugging Face.
1. Make a free account and open https://huggingface.co/mental/mental-bert-base-uncased, then accept the terms.
2. Create a read token at https://huggingface.co/settings/tokens.
3. In Colab, click the key icon on the left, add a secret named `HF_TOKEN`, paste the token, and turn on notebook access.

In [ ]:
# CELL 1: Install and log in
!pip install transformers accelerate -q

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# CELL 2: Imports, settings, load data
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          set_seed)
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                             precision_score, confusion_matrix)

MODEL_NAME = 'mental/mental-bert-base-uncased'
# For a general-purpose comparison later, change to 'bert-base-uncased' or 'roberta-base'
MAX_LEN    = 256     # posts longer than this are cut off
SEED       = 42
set_seed(SEED)

print('GPU available:', torch.cuda.is_available())

train = pd.read_csv('train_clean.csv').fillna('')
dev   = pd.read_csv('dev_clean.csv').fillna('')
print('Shapes (expect 6006 / 1000):', train.shape[0], dev.shape[0])

# BERT models want natural text, so use text_minimal (not the stopword-stripped text_clean)
label2id = {'not depressed': 0, 'depressed': 1}
y_train = train['labels'].map(label2id).values
y_dev   = dev['labels'].map(label2id).values
print('Train class counts (0=not dep, 1=dep):', np.bincount(y_train))

In [ ]:
# CELL 3: Tokenize and wrap in datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PostDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = int(self.labels[i])
        return item

train_ds = PostDataset(train['text_minimal'], y_train)
dev_ds   = PostDataset(dev['text_minimal'],   y_dev)

lengths = [len(x) for x in tokenizer(list(train['text_minimal'][:1000]))['input_ids']]
print(f'Share of posts longer than {MAX_LEN} tokens (first 1000 train posts): '
      f'{np.mean(np.array(lengths) > MAX_LEN):.1%}')

In [ ]:
# CELL 4: Model, class weights, and metrics
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Same idea as class_weight='balanced' in your sklearn models
counts  = np.bincount(y_train)
weights = torch.tensor(len(y_train) / (2 * counts), dtype=torch.float)
print('Class weights (not dep, dep):', weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        'auc':       roc_auc_score(labels, probs),
        'macro_f1':  f1_score(labels, preds, average='macro'),
        'precision': precision_score(labels, preds),
        'recall':    recall_score(labels, preds),
        'pct_flagged': preds.mean() * 100,
    }

In [ ]:
# CELL 5: Train. The best epoch is chosen by dev AUC (dev is for model selection)
args = TrainingArguments(
    output_dir='mentalbert_out',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# CELL 6: Final dev results for the best epoch, in the same format as your other models
out = trainer.predict(dev_ds)
dev_probs = softmax(out.predictions)[:, 1]
preds = (dev_probs >= 0.5).astype(int)

print('MENTALBERT ON DEV SET (cutoff 0.5)')
print(f'AUC:                   {roc_auc_score(y_dev, dev_probs):.3f}')
print(f'Macro F1:              {f1_score(y_dev, preds, average="macro"):.3f}')
print(f'Precision (depressed): {precision_score(y_dev, preds):.3f}')
print(f'Recall (depressed):    {recall_score(y_dev, preds):.3f}')
print(f'% flagged depressed:   {preds.mean() * 100:.1f}  (true share is 60.0)')

tn, fp, fn, tp = confusion_matrix(y_dev, preds).ravel()
print(f'\nTP {tp} | FN (missed) {fn} | FP (false alarms) {fp} | TN {tn}')

# Save probabilities so you can compare with LR, do bootstrap CIs, and pick cutoffs later
pd.DataFrame({'prob_depressed': dev_probs, 'label': y_dev}).to_csv('mentalbert_dev_probs.csv', index=False)
print('Saved mentalbert_dev_probs.csv')

## Cutoff sweep
The 0.5 cutoff is just a default. This shows what happens to recall and false alarms as the cutoff moves.
It is the kind of table the professor wants for the "pick a better cutoff" discussion. It is still dev only.

In [ ]:
# CELL 7: Cutoff sweep on dev
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = (dev_probs >= t).astype(int)
    rows.append({
        'cutoff': t,
        'recall (dep)': recall_score(y_dev, p),
        'precision (dep)': precision_score(y_dev, p, zero_division=0),
        'macro F1': f1_score(y_dev, p, average='macro'),
        '% flagged': p.mean() * 100,
        'missed (FN)': int(((p == 0) & (y_dev == 1)).sum()),
        'false alarms (FP)': int(((p == 1) & (y_dev == 0)).sum()),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# CELL 8: Save the fine-tuned model (so you do not retrain for the Streamlit demo)
trainer.save_model('mentalbert_best')
tokenizer.save_pretrained('mentalbert_best')

# Optional: copy to Drive so it survives the Colab session
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r mentalbert_best /content/drive/MyDrive/thesis_mentalbert